# 1. Import Dataset


We are using a dataset from Hugging Face for phishing emails: https://huggingface.co/datasets/zefang-liu/phishing-email-dataset/viewer

First, we are importing the dataset directly:

In [ ]:
! pip install -q datasets huggingface_hub

In [ ]:
from datasets import load_dataset

dataset = load_dataset("zefang-liu/phishing-email-dataset")

# 2. Preprocessing

In [ ]:
import pandas as pd

df = pd.DataFrame(dataset['train'])
df.head()

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [ ]:
df = df[["Email Text", "Email Type"]]
df.columns = ["text", "label"]
df.head()

,text,label
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,the other side of * galicismos * * galicismo *...,Safe Email
2,re : equistar deal tickets are you still avail...,Safe Email
3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [ ]:
df["label"] = df["label"].apply(lambda x: 1 if x == "Phishing Email" else 0)
df.head()

,text,label
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",0
1,the other side of * galicismos * * galicismo *...,0
2,re : equistar deal tickets are you still avail...,0
3,\nHello I am your hot lil horny toy.\n I am...,1
4,software at incredibly low prices ( 86 % lower...,1


The code below standarizes text so the model does not memorize safe IDs, accounts, urls, etc.

In [ ]:
import pandas as pd
import re
from collections import defaultdict

def clean_email_text(text):
    """Basic text cleaning for duplicate detection."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)   # remove URLs
    text = re.sub(r"[^a-z0-9\s]", " ", text)      # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()      # normalize spaces
    return text

def further_clean(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"\S+@\S+", " <EMAIL> ", text)
    text = re.sub(r"\b\d+\b", " <NUM> ", text)
    text = re.sub(r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\b", " <DATE> ", text)
    text = re.sub(r"\b(account|invoice|order|id|number)\s*\w*", r"\1 <ID>", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
df["text"] = df["text"].apply(clean_email_text)
df["text"] = df["text"].apply(further_clean)

df.head()


,text,label
0,re <NUM> <NUM> disc uniformitarianism re <NUM>...,0
1,the other side of galicismos galicismo is a sp...,0
2,re equistar deal tickets are you still availab...,0
3,hello i am your hot lil horny toy i am the one...,1
4,software at incredibly low prices <NUM> lower ...,1


In [ ]:
df = df.drop_duplicates()
df = df.dropna()
print(len(df))

17136


In [ ]:
df["text_length"] = df["text"].str.split().str.len()

df.groupby("label")["text_length"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,10691.0,547.966327,24214.441023,0.0,76.5,163.0,324.5,2502955.0
1,6445.0,263.149263,495.822992,0.0,60.0,117.0,256.0,11751.0


In [ ]:
df = df[df["text_length"] <= 500]
df = df[df["text_length"] > 20]

df.groupby("label")["text_length"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,8741.0,172.383137,119.145083,21.0,75.0,143.0,245.00,500.0
1,5306.0,142.676027,110.229270,21.0,62.0,107.0,186.75,500.0


In [ ]:
len(df)

14047

# 3. Splitting Dataset

The dataset is split between training, validation, and testing with 80/10/10 split. The training dataset is used to train models, and the validation set is used to evaluate model performances in order to compare different models before choosing one to use for testing data.

In [ ]:
from sklearn.model_selection import train_test_split

x = df["text"]
y = df["label"]

# split into train(80) and temp (20)
x_train, x_temp, y_train, y_temp = train_test_split(
    x, y,
    random_state=3,
    test_size=0.2,
    stratify=y)

print(x_train.shape)
print(x_temp.shape)

(11237,)
(2810,)


In [ ]:
x_valid, x_test, y_valid, y_test = train_test_split(
    x_temp, y_temp,
    random_state=3,
    test_size=0.5,
    stratify=y_temp)

print(x_valid.shape)
print(x_test.shape)

(1405,)
(1405,)


In [ ]:
x_train_list = x_train.to_list()
x_valid_list = x_valid.to_list()
y_train_list = y_train.to_list()
y_valid_list = y_valid.to_list()

# 4. Load Tokenizer and Model

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
train_inputs = tokenizer(
    ["classify email: " + text for text in x_train_list],
    truncation=True,
    padding=True,
    max_length=512
)

y_train_text = ["phishing" if label == 1 else "safe" for label in y_train_list]
y_valid_text = ["phishing" if label == 1 else "safe" for label in y_valid_list]

train_labels = tokenizer(
    y_train_text,
    truncation=True,
    padding=True,
    max_length=10
)

valid_inputs = tokenizer(
    ["classify email: " + text for text in x_valid_list],
    truncation=True,
    padding=True,
    max_length=512
)

valid_labels = tokenizer(
    y_valid_text,
    truncation=True,
    padding=True,
    max_length=10
)

In [ ]:
import torch

In [ ]:
class T5Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels["input_ids"][idx])
        return item

    def __len__(self):
        return len(self.labels["input_ids"])

In [ ]:
train_dataset = T5Dataset(train_inputs, train_labels)
valid_dataset = T5Dataset(valid_inputs, valid_labels)

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
100,5.004281,0.934978
200,0.641763,0.235983
300,0.248718,0.207629
400,0.168102,0.137500
500,0.128937,0.090453
600,0.118617,0.115481
700,0.093506,0.060989
800,0.070493,0.107687
900,0.074164,0.076707
1000,0.069535,0.044302


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3494, training_loss=0.21842756493813525, metrics={'train_runtime': 1909.0608, 'train_samples_per_second': 14.642, 'train_steps_per_second': 1.83, 'total_flos': 3783074034745344.0, 'train_loss': 0.21842756493813525, 'epoch': 2.0})

In [ ]:
# save model
model_path = "./t5_phishing_classifier"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)
print("Model saved tp: ", {model_path})

In [ ]:
def predict(text):
    input_text = "classify email: " + text
    input_ids = tokenizer.encode(input_text, return_tensors="pt", truncation=True)

    outputs = model.generate(input_ids)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def predict(text):
    input_text = "classify email: " + text

    input_ids = tokenizer.encode(
        input_text,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    outputs = model.generate(input_ids)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
preds = []
true = []

for i in range(len(x_valid_list)):
    pred = predict(x_valid_list[i])

    # convert text → label
    pred_label = 1 if pred.strip().lower() == "phishing" else 0

    preds.append(pred_label)
    true.append(y_valid_list[i])

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print(f"Accuracy score: {accuracy_score(true, preds)}")
print(classification_report(true, preds))

              precision    recall  f1-score   support

           0       0.97      0.96      0.97      1104
           1       0.94      0.95      0.94       644

    accuracy                           0.96      1748
   macro avg       0.95      0.95      0.95      1748
weighted avg       0.96      0.96      0.96      1748



In [ ]:
!zip -r t5_phishing_classifier.zip t5_phishing_classifier

  adding: t5_phishing_classifier/ (stored 0%)
  adding: t5_phishing_classifier/model.safetensors (deflated 12%)
  adding: t5_phishing_classifier/tokenizer.json (deflated 79%)
  adding: t5_phishing_classifier/config.json (deflated 63%)
  adding: t5_phishing_classifier/tokenizer_config.json (deflated 83%)
  adding: t5_phishing_classifier/generation_config.json (deflated 29%)
